Notes(What needs to be done): 
1. for claim frequency Check var/ mean ratio(Coefficent of variation) to narrow candidates (initial list: Poisson, Negative Binomial, Zero-Inflated Poisson, Zero inflated negative binomial)
2. check what predictors are available at underwriting time(not rating)
etc

# 04 - Frequency Distribution

**Purpose:** Identify the distribution that claim counts follow, for each of the
four coverages. Candidate families are narrowed by the variance-to-mean ratio,
then ranked by AIC and BIC on maximum likelihood fits, and the winner is
confirmed with a chi-square goodness-of-fit test where the counts support one.
No predictors are involved; this is the marginal distribution only.

**Input:** `data/processed/frequency_data.csv`
**Output:** None. Conclusions carried into 06_FrequencyModeling.
**Depends on:** 02_Cleaned_Data

**Candidates:** Poisson, Negative Binomial, Zero-Inflated Poisson,
Zero-Inflated Negative Binomial

analyze var / mean and get candidate distibutions

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
PROC = ROOT / "data" / "processed"
FIGS = ROOT / "outputs" / "figures"

freq = pd.read_csv(PROC / "frequency_data.csv")

COVERAGES = ['Personal Property', 'Additional Living Expense',
             'Guest Medical', 'Liability']

fq = {c: freq[freq.coverage == c].reset_index(drop=True) for c in COVERAGES}

{c: len(d) for c, d in fq.items()}

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))

for ax, cov in zip(axes.flat, COVERAGES):
    d = fq[cov].claim_count
    d.value_counts().sort_index().plot.bar(ax=ax, rot=0, color='#1f77b4')
    ax.set_title(cov)
    ax.set_xlabel('claims per student')
    ax.set_ylabel('students')

plt.tight_layout()

Analyze BIC and AIC of each proposed distribution, an optimizer is used to get a numerical estimate of the MLE of each distribution

In [ ]:
from scipy import stats, optimize

y    = fq['Personal Property'].claim_count.values
n    = len(y)
ybar = y.mean()

results = {}

def fit(nll, x0, k):
    r = optimize.minimize(nll, x0, method='Nelder-Mead',
                          options={'xatol':1e-8, 'fatol':1e-8, 'maxiter':5000})
    return -r.fun, k

results['Poisson'] = (stats.poisson.logpmf(y, ybar).sum(), 1)

def nll_zip(t):
    lam = np.exp(t[0]); pi = 1/(1+np.exp(-t[1]))
    return -np.where(y == 0,
                     np.log(pi + (1-pi)*np.exp(-lam)),
                     np.log1p(-pi) + stats.poisson.logpmf(y, lam)).sum()
results['ZIP'] = fit(nll_zip, [np.log(ybar), -2.0], 2)

def nb_lpmf(y, mu, a):
    r = 1/a
    return stats.nbinom.logpmf(y, r, r/(r+mu))

def nll_nb(t):
    return -nb_lpmf(y, np.exp(t[0]), np.exp(t[1])).sum()
results['NegBinomial'] = fit(nll_nb, [np.log(ybar), np.log(0.5)], 2)

def nll_zinb(t):
    mu, a = np.exp(t[0]), np.exp(t[1]); pi = 1/(1+np.exp(-t[2]))
    r  = 1/a
    p0 = stats.nbinom.pmf(0, r, r/(r+mu))
    return -np.where(y == 0,
                     np.log(pi + (1-pi)*p0),
                     np.log1p(-pi) + nb_lpmf(y, mu, a)).sum()
results['ZINB'] = fit(nll_zinb, [np.log(ybar), np.log(0.5), -2.0], 3)

tbl = pd.DataFrame(
    [(name, ll, k, 2*k - 2*ll, k*np.log(n) - 2*ll)
     for name, (ll, k) in results.items()],
    columns=['model', 'loglik', 'k', 'AIC', 'BIC']
).sort_values('BIC', ascending = False)

tbl.round(2)

Based on the above table, possoin has the lowest bic and thus we will assume that personal property claim frequency follows a poisson distribution

Additional Living Expense

In [ ]:
from scipy import stats, optimize

y    = fq['Additional Living Expense'].claim_count.values
n    = len(y)
ybar = y.mean()

results = {}

def fit(nll, x0, k):
    r = optimize.minimize(nll, x0, method='Nelder-Mead',
                          options={'xatol':1e-8, 'fatol':1e-8, 'maxiter':5000})
    return -r.fun, k

results['Poisson'] = (stats.poisson.logpmf(y, ybar).sum(), 1)

def nll_zip(t):
    lam = np.exp(t[0]); pi = 1/(1+np.exp(-t[1]))
    return -np.where(y == 0,
                     np.log(pi + (1-pi)*np.exp(-lam)),
                     np.log1p(-pi) + stats.poisson.logpmf(y, lam)).sum()
results['ZIP'] = fit(nll_zip, [np.log(ybar), -2.0], 2)

def nb_lpmf(y, mu, a):
    r = 1/a
    return stats.nbinom.logpmf(y, r, r/(r+mu))

def nll_nb(t):
    return -nb_lpmf(y, np.exp(t[0]), np.exp(t[1])).sum()
results['NegBinomial'] = fit(nll_nb, [np.log(ybar), np.log(0.5)], 2)

def nll_zinb(t):
    mu, a = np.exp(t[0]), np.exp(t[1]); pi = 1/(1+np.exp(-t[2]))
    r  = 1/a
    p0 = stats.nbinom.pmf(0, r, r/(r+mu))
    return -np.where(y == 0,
                     np.log(pi + (1-pi)*p0),
                     np.log1p(-pi) + nb_lpmf(y, mu, a)).sum()
results['ZINB'] = fit(nll_zinb, [np.log(ybar), np.log(0.5), -2.0], 3)

tbl = pd.DataFrame(
    [(name, ll, k, 2*k - 2*ll, k*np.log(n) - 2*ll)
     for name, (ll, k) in results.items()],
    columns=['model', 'loglik', 'k', 'AIC', 'BIC']
).sort_values('BIC', ascending = False)

tbl.round(2)

Guest Medical 

In [ ]:
from scipy import stats, optimize

y    = fq['Guest Medical'].claim_count.values
n    = len(y)
ybar = y.mean()

results = {}

def fit(nll, x0, k):
    r = optimize.minimize(nll, x0, method='Nelder-Mead',
                          options={'xatol':1e-8, 'fatol':1e-8, 'maxiter':5000})
    return -r.fun, k

results['Poisson'] = (stats.poisson.logpmf(y, ybar).sum(), 1)

def nll_zip(t):
    lam = np.exp(t[0]); pi = 1/(1+np.exp(-t[1]))
    return -np.where(y == 0,
                     np.log(pi + (1-pi)*np.exp(-lam)),
                     np.log1p(-pi) + stats.poisson.logpmf(y, lam)).sum()
results['ZIP'] = fit(nll_zip, [np.log(ybar), -2.0], 2)

def nb_lpmf(y, mu, a):
    r = 1/a
    return stats.nbinom.logpmf(y, r, r/(r+mu))

def nll_nb(t):
    return -nb_lpmf(y, np.exp(t[0]), np.exp(t[1])).sum()
results['NegBinomial'] = fit(nll_nb, [np.log(ybar), np.log(0.5)], 2)

def nll_zinb(t):
    mu, a = np.exp(t[0]), np.exp(t[1]); pi = 1/(1+np.exp(-t[2]))
    r  = 1/a
    p0 = stats.nbinom.pmf(0, r, r/(r+mu))
    return -np.where(y == 0,
                     np.log(pi + (1-pi)*p0),
                     np.log1p(-pi) + nb_lpmf(y, mu, a)).sum()
results['ZINB'] = fit(nll_zinb, [np.log(ybar), np.log(0.5), -2.0], 3)

tbl = pd.DataFrame(
    [(name, ll, k, 2*k - 2*ll, k*np.log(n) - 2*ll)
     for name, (ll, k) in results.items()],
    columns=['model', 'loglik', 'k', 'AIC', 'BIC']
).sort_values('BIC', ascending = False)

tbl.round(2)

Liability

In [ ]:
from scipy import stats, optimize

y    = fq['Liability'].claim_count.values
n    = len(y)
ybar = y.mean()

results = {}

def fit(nll, x0, k):
    r = optimize.minimize(nll, x0, method='Nelder-Mead',
                          options={'xatol':1e-8, 'fatol':1e-8, 'maxiter':5000})
    return -r.fun, k

results['Poisson'] = (stats.poisson.logpmf(y, ybar).sum(), 1)

def nll_zip(t):
    lam = np.exp(t[0]); pi = 1/(1+np.exp(-t[1]))
    return -np.where(y == 0,
                     np.log(pi + (1-pi)*np.exp(-lam)),
                     np.log1p(-pi) + stats.poisson.logpmf(y, lam)).sum()
results['ZIP'] = fit(nll_zip, [np.log(ybar), -2.0], 2)

def nb_lpmf(y, mu, a):
    r = 1/a
    return stats.nbinom.logpmf(y, r, r/(r+mu))

def nll_nb(t):
    return -nb_lpmf(y, np.exp(t[0]), np.exp(t[1])).sum()
results['NegBinomial'] = fit(nll_nb, [np.log(ybar), np.log(0.5)], 2)

def nll_zinb(t):
    mu, a = np.exp(t[0]), np.exp(t[1]); pi = 1/(1+np.exp(-t[2]))
    r  = 1/a
    p0 = stats.nbinom.pmf(0, r, r/(r+mu))
    return -np.where(y == 0,
                     np.log(pi + (1-pi)*p0),
                     np.log1p(-pi) + nb_lpmf(y, mu, a)).sum()
results['ZINB'] = fit(nll_zinb, [np.log(ybar), np.log(0.5), -2.0], 3)

tbl = pd.DataFrame(
    [(name, ll, k, 2*k - 2*ll, k*np.log(n) - 2*ll)
     for name, (ll, k) in results.items()],
    columns=['model', 'loglik', 'k', 'AIC', 'BIC']
).sort_values('BIC', ascending = False)

tbl.round(2)